# 19 - DNA-based CIDER T-cell Score vs RNA: Response Discrimination by Cohort

**Purpose**: Supplementary figure checking that the DNA-based CIDER T-cell
Score's ICI responder-discrimination signal is consistent across cancer
types when compared against four RNA-based metrics -- the primary T-cell
signature (CD3D/CD3E/CD3G, `tcell_tpm` -- Braun et al. 2021 *Cancer Cell*,
see `2_rna.ipynb`), the CD8 Effector (tGE8) signature (`rna_teff_score`),
PD-L1 (CD274) bulk RNA expression (`rna_pdl1_score` -- a transcript-level
proxy for the IHC-based PD-L1 Tumor Proportion Score/Combined Positive
Score), and a second, independently-sourced T-cell presence signature from
the Human Cell Landscape atlas (CD3D/CD3E/CD3G/CD2/TRAC,
`rna_tcell_hcl_score` -- Han, X., Zhou, Z., Fei, L. et al. "Construction of
a human cell landscape at single-cell level." *Nature* 581, 303-309, 2020).

**Input:** `cpi_ready.rds`
**Output:** `figs/supplement_dna_vs_rna_by_cohort.rds`/`.png` -- one panel
per metric (DNA-based CIDER T-cell Score, RNA T-cell Signature, RNA T-cell
Effector, RNA PD-L1, RNA T-cell Signature (HCL)), each built with
`response_plot()` and `cohort_broad_go` exactly as Figure 3's main response
panel and `17_response_by_read_confidence.ipynb` are: a pooled
"ICI-Treated" bar plus a per-cancer-type breakdown (Section 2 below).

**Also in this notebook:** a standalone forest-plot analysis (Section 1)
fits each metric's own regression coefficient across all four endpoints
(BOR, DCB, PFS, OS), both ways -- each RNA metric adjusted for CIDER, and
CIDER adjusted for each RNA metric -- since CIDER and the RNA metrics are
correlated (r ~ 0.7-0.85) and it isn't predictable in advance which signal
would attenuate the other. Kept in the notebook as a reference/exploratory
check (run that cell directly); it isn't saved to `figs/`.


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

# Alias: shared OS/PFS/BOR/DCB palette from themes.r, kept under this
# notebook's existing local name so call-sites below are unchanged.
rna_endpoint_colors <- endpoint_colors

library(ggpubr)
library(patchwork)
library(survival)  # Surv()/coxph() for the standalone cross-adjustment OS/PFS models (Section 1)

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

# 0 - Read data

In [ ]:
# cpi_ready.rds stores one row per (sampleId, gp) *and* a duplicated copy of
# every row with cohort_broad overwritten to "ICI-Treated" (4_base.ipynb, for
# Figure 3's pooled-vs-per-cancer-type panel). This figure pools every
# ICI-treated patient regardless of cancer type and doesn't use cohort_broad
# at all, so that duplication would silently double-count every patient here
# -- drop it first (same defusing pattern as 18_msi_response.ipynb), then
# restrict to the T-cell row (RNA columns are sample-level and identical
# across every gp row for a given sample, same as 4_cider_validation.ipynb).
ready <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(cohort_broad != "ICI-Treated", gp == "T-cell")

# 1 - Standalone: CIDER vs RNA T-cell signatures predicting ICI response and survival, with/without cross-adjustment

Not saved to `figs/` -- kept as a standalone exploratory analysis; run the
cell below directly for the results. (This standalone analysis only covers
the original two RNA metrics -- T-cell Signature and Effector -- PD-L1 and
the HCL T-cell signature were not added here; see the notebook intro.)

Fits each metric's own regression coefficient for all four endpoints (BOR,
DCB logistic; PFS, OS Cox PH) in both directions: each RNA metric
univariate and adjusted for CIDER, AND CIDER univariate and adjusted for
each RNA metric in turn. CIDER and the RNA metrics are correlated (r ~
0.7-0.85 in this cohort -- see the printed collinearity/VIF table), so
cross-adjustment can attenuate either side depending on which one carries
more of the outcome-relevant signal that isn't shared with the other --
both directions are fit explicitly rather than assumed. The CIDER T-cell
Score is log2(x+1)-transformed, unstandardized (same convention as
`5_models.ipynb`/`15_ici_specificity.ipynb`); both RNA comparator metrics
are log2(x+1)-transformed AND standardized.
Population is fit per-model (drops NA only for the columns that model
needs), so N varies slightly by endpoint.


In [ ]:
# ── Standalone: CIDER vs RNA T-cell signatures predicting ICI response and survival, with/without cross-adjustment ──
# cider_log2 = log2(x+1), unstandardized -- same convention used for the
# CIDER score throughout the paper (5_models.ipynb, 15_ici_specificity.ipynb).
# scaled_tcell/scaled_teff = log2(x+1), then standardized (scale()) -- these
# RNA comparator metrics aren't part of that CIDER-specific convention.
# Computed once on the full `ready` population
# so the scaling denominator (mean/SD) isn't itself endpoint-dependent.
# Each individual model below then drops NA only for the columns IT needs
# (outcome + metric [+ adjustment covariate, if any]) -- the same per-model
# drop_na pattern used by fit_logistic()/fit_cox() elsewhere in the paper --
# rather than requiring one population shared across all four endpoints,
# since the set of patients with known BOR/DCB isn't the same as the set
# with known OS/PFS, and intersecting them would needlessly shrink N.
rna_forest_data <- ready |>
  mutate(
    # log2(x+1), unstandardized -- matches the convention used by every
    # forest plot in the paper (0_prepare/5_models.ipynb,
    # 15_ici_specificity.ipynb, 20_ici_specificity_main.ipynb); the RNA
    # comparator metrics below stay standardized since they aren't part of
    # that cross-figure CIDER convention.
    cider_log2   = log2(cider_score_depth_adj + 1),
    scaled_tcell = as.vector(scale(log2(tcell_tpm + 1))),
    scaled_teff  = as.vector(scale(log2(rna_teff_score + 1))),
    response     = as.integer(response),
    dcb          = as.integer(dcb),
    os_months    = osDays / 30.44,
    pfs_months   = pfsEventDays / 30.44
  )

MIN_N_RNA      <- 20
MIN_EVENTS_RNA <- 5

# Display label for each possible adjustment covariate (NA = univariate).
ADJUST_LABELS <- c(
  cider_log2 = "Adjusted for CIDER",
  scaled_tcell = "Adjusted for RNA Signature",
  scaled_teff  = "Adjusted for RNA Effector"
)
covariate_label <- function(adjust_col) {
  if (is.na(adjust_col)) "Unadjusted" else unname(ADJUST_LABELS[adjust_col])
}

# Fits outcome ~ metric (univariate, adjust_col = NA) or outcome ~ metric +
# adjust_col, returning the metric's own coefficient as an OR with 95% Wald
# CI. Only the metric term is extracted -- for an adjusted model this isn't
# meant to characterise the adjustment covariate's own coefficient, just
# whether the metric of interest holds up once that covariate is in the
# model. Generalised over which covariate (if any) is added so the same
# function fits both directions: an RNA metric adjusted for CIDER, and
# CIDER adjusted for an RNA metric.
fit_rna_logistic <- function(df, outcome_col, metric_col, adjust_col,
                             metric_label, endpoint_label) {
  covars <- if (is.na(adjust_col)) metric_col else c(metric_col, adjust_col)
  d <- df |> tidyr::drop_na(dplyr::all_of(c(outcome_col, covars)))
  outcome <- d[[outcome_col]]
  if (nrow(d) < MIN_N_RNA || sum(outcome, na.rm = TRUE) < MIN_EVENTS_RNA ||
      length(unique(outcome)) < 2) {
    message("Skipping ", metric_label, " / ", endpoint_label,
            " (", covariate_label(adjust_col), ", n=", nrow(d), ")")
    return(NULL)
  }
  fit <- tryCatch(
    suppressWarnings(glm(reformulate(covars, outcome_col), data = d, family = binomial())),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == metric_col) |>
    dplyr::mutate(
      conf.low   = exp(estimate - 1.96 * std.error),
      conf.high  = exp(estimate + 1.96 * std.error),
      estimate   = exp(estimate),
      metric     = metric_label,
      covariates = covariate_label(adjust_col),
      endpoint   = endpoint_label,
      n          = nrow(d)
    )
}

# Same idea as fit_rna_logistic() above, but Cox PH for the survival
# endpoints (OS, PFS).
fit_rna_cox <- function(df, time_col, event_col, metric_col, adjust_col,
                        metric_label, endpoint_label) {
  covars <- if (is.na(adjust_col)) metric_col else c(metric_col, adjust_col)
  d <- df |>
    tidyr::drop_na(dplyr::all_of(c(time_col, event_col, covars))) |>
    dplyr::filter(.data[[time_col]] > 0)
  events <- d[[event_col]]
  if (nrow(d) < MIN_N_RNA || sum(events, na.rm = TRUE) < MIN_EVENTS_RNA) {
    message("Skipping ", metric_label, " / ", endpoint_label,
            " (", covariate_label(adjust_col), ", n=", nrow(d),
            ", events=", sum(events, na.rm = TRUE), ")")
    return(NULL)
  }
  fmla <- as.formula(paste0("Surv(", time_col, ", ", event_col, ") ~ ", paste(covars, collapse = " + ")))
  fit  <- tryCatch(suppressWarnings(survival::coxph(fmla, data = d)), error = function(e) NULL)
  if (is.null(fit)) return(NULL)
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == metric_col) |>
    dplyr::mutate(
      conf.low   = exp(estimate - 1.96 * std.error),
      conf.high  = exp(estimate + 1.96 * std.error),
      estimate   = exp(estimate),
      metric     = metric_label,
      covariates = covariate_label(adjust_col),
      endpoint   = endpoint_label,
      n          = nrow(d)
    )
}

# RNA metrics: univariate, and adjusted for CIDER (16 models: 2 metrics x
# 4 endpoints x 2 adjust_col options).
rna_model_specs <- tidyr::expand_grid(
  metric_col = c("scaled_tcell", "scaled_teff"),
  endpoint   = c("BOR", "DCB", "PFS", "OS"),
  adjust_col = c(NA_character_, "cider_log2")
) |>
  dplyr::mutate(
    metric_label = dplyr::recode(metric_col,
      scaled_tcell = "RNA T-cell Signature",
      scaled_teff  = "RNA T-cell Effector (tGE8)"
    )
  )

# CIDER: univariate, and adjusted for EACH RNA metric in turn (12 models:
# 1 metric x 4 endpoints x 3 adjust_col options). This is the reverse
# direction of the block above -- whether CIDER's own effect holds up once
# an RNA metric is in the model, rather than only testing whether the RNA
# metric holds up once CIDER is in the model. Which direction attenuates
# more isn't predictable from the CIDER-RNA correlation alone (see the
# collinearity diagnostic below) -- it depends on which predictor carries
# more of the outcome-relevant signal that ISN'T shared with the other, so
# both directions are fit explicitly rather than assumed.
cider_model_specs <- tidyr::expand_grid(
  metric_col = "cider_log2",
  endpoint   = c("BOR", "DCB", "PFS", "OS"),
  adjust_col = c(NA_character_, "scaled_tcell", "scaled_teff")
) |>
  dplyr::mutate(metric_label = "CIDER T-cell Score")

all_model_specs <- dplyr::bind_rows(rna_model_specs, cider_model_specs)

# ── Collinearity diagnostic ──────────────────────────────────────────────────
# CIDER and the RNA metrics are correlated (r ~ 0.7-0.85 in this
# cohort) -- adding one as a covariate for the other redistributes shared
# variance between two correlated predictors, which can attenuate, leave
# essentially unchanged, or even widen either side's adjusted CI depending
# on how much of its signal is "unique" (not already explained by the
# other). That's expected behaviour under collinearity, not evidence the
# adjustment isn't being applied. VIF = 1/(1-r^2) quantifies the resulting
# SE inflation (~sqrt(VIF)x wider CI than the univariate model, all else
# equal) -- symmetric in both directions, since VIF depends only on r.
rna_collinearity <- tibble::tibble(
  metric = c("RNA T-cell Signature", "RNA T-cell Effector (tGE8)"),
  r      = c(cor(rna_forest_data$cider_log2, rna_forest_data$scaled_tcell, use = "complete.obs"),
             cor(rna_forest_data$cider_log2, rna_forest_data$scaled_teff,  use = "complete.obs"))
) |>
  dplyr::mutate(vif = 1 / (1 - r^2), se_inflation_x = sqrt(vif))

message("\nCIDER vs RNA metric collinearity (symmetric -- drives how much cross-adjustment shifts either side's coefficient):")
print(rna_collinearity)

rna_forest_results <- purrr::pmap_dfr(
  all_model_specs,
  function(metric_col, endpoint, adjust_col, metric_label) {
    if (endpoint %in% c("BOR", "DCB")) {
      outcome_col <- if (endpoint == "BOR") "response" else "dcb"
      fit_rna_logistic(rna_forest_data, outcome_col, metric_col, adjust_col,
                       metric_label, endpoint)
    } else {
      time_col  <- if (endpoint == "OS") "os_months"  else "pfs_months"
      event_col <- if (endpoint == "OS") "osEvent"    else "pfsEvent"
      fit_rna_cox(rna_forest_data, time_col, event_col, metric_col, adjust_col,
                 metric_label, endpoint)
    }
  }
) |>
  dplyr::mutate(
    metric     = factor(metric, levels = c("CIDER T-cell Score", "RNA T-cell Signature", "RNA T-cell Effector (tGE8)")),
    covariates = factor(covariates, levels = c("Unadjusted", "Adjusted for CIDER",
                                                "Adjusted for RNA Signature", "Adjusted for RNA Effector")),
    endpoint   = factor(endpoint, levels = c("BOR", "DCB", "PFS", "OS")),
    # 28 tests total in this panel (16 RNA-adjusted-for-CIDER + 12
    # CIDER-adjusted-for-RNA/univariate) -- plain Wald p-values are reported
    # directly rather than FDR-corrected, and the caption says so explicitly
    # so the stars aren't mistaken for the FDR-controlled stars used
    # elsewhere in the paper's larger forest plots.
    sig_star = dplyr::case_when(
      p.value < 0.001 ~ "***",
      p.value < 0.01  ~ "**",
      p.value < 0.05  ~ "*",
      TRUE            ~ ""
    )
  )

message("\nStandalone cross-adjustment analysis: N per model (varies by endpoint -- BOR/DCB and OS/PFS annotation aren't the same patient set):")
print(rna_forest_results |> dplyr::select(metric, endpoint, covariates, n, estimate, conf.low, conf.high, p.value))

rna_covariate_shapes <- c(
  "Unadjusted"                 = 16,
  "Adjusted for CIDER"         = 17,
  "Adjusted for RNA Signature" = 15,
  "Adjusted for RNA Effector"  = 18
)
# rna_endpoint_colors (OS/PFS/BOR/DCB) is aliased to the shared endpoint_colors
# palette from helpers/themes.r -- see alias near the top of the notebook.

p_rna_forest <- rna_forest_results |>
  ggplot(aes(x = estimate, y = metric, colour = endpoint, shape = covariates)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE) +
  geom_pointrange(
    aes(xmin = conf.low, xmax = conf.high),
    position = position_dodge(width = 0.8), size = 0.6, fatten = 3
  ) +
  geom_text(
    aes(x = conf.high, label = sig_star),
    position = position_dodge(width = 0.8),
    hjust = -0.3, size = 3, colour = COL_INK, show.legend = FALSE
  ) +
  scale_colour_manual(values = rna_endpoint_colors, name = "Endpoint") +
  scale_shape_manual(values = rna_covariate_shapes, name = "Model") +
  scale_x_log10() +
  go_theme +
  theme(
    plot.title    = element_text(size = 13, face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5),
    legend.position = "right"
  ) +
  labs(
    x        = "OR / HR (95% CI, log scale)",
    y        = NULL,
    title    = "CIDER and RNA T-cell Signatures: Effect Sizes With and Without Cross-Adjustment",
    subtitle = "ICI-treated patients with CIDER T-cell Score and RNA-seq (N varies by endpoint, see printed table above)",
    caption  = paste(
      "CIDER T-cell Score is log2(x+1)-transformed (unstandardized); both RNA metrics are log2(x+1)-transformed and standardized (mean 0, SD 1) before fitting.",
      "Stars: * p<0.05  ** p<0.01  *** p<0.001 (unadjusted Wald p-values -- 28 tests here, not FDR-corrected).",
      "Each point is the labelled metric's OWN coefficient; \"Adjusted for X\" means X was added as a covariate in that model.",
      "CIDER and the RNA metrics are correlated (r ~ 0.7-0.85) -- cross-adjustment tests which signal holds up, not just CIDER's.",
      sep = "\n"
    )
  )

set_plot_dims(10, 5); p_rna_forest

# 2 - Response discrimination by cohort

`figs/supplement_dna_vs_rna_by_cohort.rds`/`.png` -- showing all five
metrics (DNA-based CIDER T-cell Score, RNA T-cell Signature, RNA T-cell
Effector [tGE8], RNA PD-L1, RNA T-cell Signature [HCL]) with cohort on the
x-axis -- one panel per metric, each built with `response_plot()` and
`cohort_broad_go` exactly as Figure 3's main response panel and
`17_response_by_read_confidence.ipynb` are: a pooled "ICI-Treated" bar plus
a per-cancer-type breakdown, so the responder-discrimination signal can be
checked for consistency across cohorts rather than assuming it's uniform.

Population is restricted to patients with all five metrics + known
response present, so all five panels and every cohort describe the same
patients -- read fresh from `cpi_ready.rds` (rather than the `ready` object
from Section 0, which has the `cohort_broad == "ICI-Treated"` duplicate
rows already dropped) so that duplicate is available for
`cohort_broad_go`'s pooled bar.


In [ ]:
# cpi_ready.rds carries both the real-cohort row and a duplicated copy with
# cohort_broad overwritten to "ICI-Treated" for every ICI-treated patient
# (4_base.ipynb) -- `ready` (Section 0) deliberately drops that duplicate
# since this notebook otherwise pools every ICI-treated patient regardless
# of cancer type. Here we want the opposite: both copies, landing in
# separate cohort_broad_go bins (pooled "ICI-Treated" + each real cancer
# type), same mechanism Figure 3 and 17_response_by_read_confidence.ipynb
# use for their pooled-plus-per-cohort panels.
#
# Same common-population restriction as box_data (Section 2): all five
# metrics and responder present, so every cohort panel below describes the
# same patient set.
cohort_box_data <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(
    gp == "T-cell",
    !is.na(cider_score_depth_adj), !is.na(tcell_tpm),
    !is.na(rna_teff_score), !is.na(rna_pdl1_score),
    !is.na(rna_tcell_hcl_score), !is.na(responder)
  )

# One response_plot() panel per metric, cohort_broad_go on the x-axis
# (pooled "ICI-Treated" + per-cancer-type, x_dodge = 2 to keep the cohort
# labels readable). response_plot()'s y-axis label only recognises the
# three built-in CIDER y_col names (helpers/themes.r) -- overridden below
# with labs(y = ...) for all five panels so the axis labels match the
# notebook intro's wording exactly (including the DNA-based CIDER rename above).
cohort_response_panels <- list(
  response_plot(cohort_box_data, "T-cell", "cohort_broad_go",
                title = "DNA-based CIDER T-cell Score",
                y_col = "cider_score_depth_adj", x_dodge = 2) +
    ggplot2::labs(y = "DNA-based CIDER T-cell Score"),
  response_plot(cohort_box_data, "T-cell", "cohort_broad_go",
                title = "RNA T-cell Signature",
                y_col = "tcell_tpm", x_dodge = 2) +
    ggplot2::labs(y = "RNA T-cell Expression (mean AdjTPM)"),
  response_plot(cohort_box_data, "T-cell", "cohort_broad_go",
                title = "RNA T-cell Effector (tGE8)",
                y_col = "rna_teff_score", x_dodge = 2) +
    ggplot2::labs(y = "RNA T-cell Effector Score (mean AdjTPM)"),
  response_plot(cohort_box_data, "T-cell", "cohort_broad_go",
                title = "RNA PD-L1 (CD274)",
                y_col = "rna_pdl1_score", x_dodge = 2) +
    ggplot2::labs(y = "RNA PD-L1 Expression (mean AdjTPM)"),
  response_plot(cohort_box_data, "T-cell", "cohort_broad_go",
                title = "RNA T-cell Signature (HCL)",
                y_col = "rna_tcell_hcl_score", x_dodge = 2) +
    ggplot2::labs(y = "RNA T-cell Expression, HCL panel (mean AdjTPM)")
)

# 5 panels at ncol = 2 -> exactly 3 rows, same ~4/row convention used
# elsewhere (height = 12).
p_dna_rna_response_by_cohort <- patchwork::wrap_plots(cohort_response_panels, ncol = 2, guides = "collect") +
  figure_annotation("DNA-based CIDER T-cell Score vs RNA: Response Discrimination by Cohort")

set_plot_dims(14, 12); p_dna_rna_response_by_cohort

saveRDS(p_dna_rna_response_by_cohort, paste0(CIDER_PROJ_DIR, "figs/supplement_dna_vs_rna_by_cohort.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_dna_vs_rna_by_cohort.png"),
       p_dna_rna_response_by_cohort, width = 14, height = 12, dpi = 300)
message("Saved: supplement_dna_vs_rna_by_cohort.rds and supplement_dna_vs_rna_by_cohort.png")
